## <b><font color='darkblue'>Introduction</font></b>
<b><font size='3ptx'>([source](https://machinelearningmastery.com/7-numpy-tricks-to-vectorize-your-code/)) You’ve written Python that processes data in a loop. It’s clear, it’s correct, and it’s unusably slow on real-world data sizes</font></b>. The problem isn’t your algorithm; it’s that for loops in Python execute at interpreter speed, which means every iteration pays the overhead cost of Python’s dynamic type checking and memory management.

<b><font size='3ptx'>[NumPy](https://numpy.org/) helps solve this bottleneck</font></b>. It wraps highly optimized C and Fortran libraries that can process entire arrays in single operations, bypassing Python’s overhead completely. But you need to write your code differently — and express it as vectorized operations — to access that speed. The shift requires a different way of thinking. Instead of “loop through and check each value,” you think “select elements matching a condition.” Instead of nested iteration, you think in array dimensions and broadcasting.

<b><font size='3ptx'>This article walks through 7 vectorization techniques that eliminate loops from numerical code</font></b>. Each one addresses a specific pattern where developers typically reach for iteration, showing you how to reformulate the problem in array operations instead. The result is code that runs much (much) faster and often reads more clearly than the loop-based version.

In [2]:
import numpy as np

## <b><font color='darkblue'>1. Boolean Indexing Instead of Conditional Loops</font></b>
<b><font size='3ptx'>You need to filter or modify array elements based on conditions</font></b>. The instinct is to loop through and check each one.

In [5]:
data = np.array(range(10))
data

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

Naive way:

In [7]:
result = []
for x in data:
    if x % 2 == 0:
        result.append(x * 2)
    else:
        result.append(x)
        
result = np.array(result)
print(result)

[ 0  1  4  3  8  5 12  7 16  9]


Here’s the vectorized approach:

In [9]:
result = data.copy()
result[data % 2 == 0] *= 2
print(result)

[ 0  1  4  3  8  5 12  7 16  9]


## <b><font color='darkblue'>2. Broadcasting for Implicit Loops</font></b>
<b><font size='3ptx'>Sometimes you want to combine arrays of different shapes, maybe adding a row vector to every row of a matrix</font></b>. The loop-based approach requires explicit iteration. The naive way:

In [16]:
matrix = np.array([
    [0, 1, 2, 3, 4],
    [5, 6, 7, 8, 9],
])
row_means = np.mean(matrix, axis=1)
row_means

array([2., 7.])

In [17]:
centered = np.zeros_like(matrix)
centered

array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

In [18]:
for i in range(matrix.shape[0]):
    centered[i] = matrix[i] - row_means[i]

centered

array([[-2, -1,  0,  1,  2],
       [-2, -1,  0,  1,  2]])

Vectorized approach:

In [19]:
matrix = np.array([
    [0, 1, 2, 3, 4],
    [5, 6, 7, 8, 9],
])
row_means = np.mean(matrix, axis=1, keepdims=True)
row_means

array([[2.],
       [7.]])

In [20]:
centered = matrix - row_means
centered

array([[-2., -1.,  0.,  1.,  2.],
       [-2., -1.,  0.,  1.,  2.]])

Setting `keepdims=True` keeps `row_means` as shape `(1000, 1)`, not `(1000,)`. <b>When you subtract, NumPy automatically stretches this column vector across all columns of the matrix</b>. The shapes don’t match, but NumPy makes them compatible by repeating values along singleton dimensions.

## <b><font color='darkblue'>3. `np.where()` for Vectorized If-Else</font></b>
When you need different calculations for different elements based on conditions, you’ll need to write branching logic inside loops.

In [25]:
temps = np.array([0, 15, -1, 33, 5, 30, 18, -10])

In [26]:
classifications = []
for t in temps:
    if t < 0:
        classifications.append('freezing')
    elif t < 20:
        classifications.append('cool')
    else:
        classifications.append('warm')

classifications

['cool', 'cool', 'freezing', 'warm', 'cool', 'warm', 'cool', 'freezing']

Here’s the vectorized approach:

In [29]:
classifications = np.select(
    [temps < 0, temps < 20, temps >= 20],
    ['freezing', 'cool', 'warm'],
    default='unknown' # Added a string default value
)

classifications.tolist()

['cool', 'cool', 'freezing', 'warm', 'cool', 'warm', 'cool', 'freezing']

For simple splits, `np.where()` is cleaner:

In [30]:
scores = np.array([45, 60, 78, 55, 95])
results = np.where(scores >= 60, 'pass', 'fail')
results

array(['fail', 'pass', 'pass', 'fail', 'pass'], dtype='<U4')

`np.where(condition, x, y)` returns elements from `x` where condition is True, from y elsewhere. `np.select()` extends this to multiple conditions. It checks each condition in order and returns the corresponding value from the second list.

## <b><font color='darkblue'>4. Better Indexing for Lookup Operations</font></b>
Suppose you have indices and need to gather elements from multiple positions. You’ll often reach for dictionary lookups in loops, or worse, nested searches.

In [34]:
# Slow: Loop-based gathering
lookup_table = np.array([10, 20, 30, 40, 50])
indices = np.array([0, 2, 3, 4, 1])
results = []
for idx in indices:
    results.append(lookup_table[idx])
    
results = np.array(results)
results

array([10, 30, 40, 50, 20])

Here’s the vectorized approach:

In [35]:
results = lookup_table[indices]
results

array([10, 30, 40, 50, 20])

When you index an array with another array of integers, NumPy pulls out elements at those positions. This works in multiple dimensions too:

In [38]:
matrix = np.arange(20).reshape(4, 5)
matrix

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19]])

In [39]:
row_indices = np.array([0, 2, 3])
col_indices = np.array([1, 3, 4])
values = matrix[row_indices, col_indices]  # Gets matrix[0,1], matrix[2,3], matrix[3,4]
values

array([ 1, 13, 19])

## <b><font color='darkblue'>5. `np.vectorize()` for Custom Functions</font></b>
You have a function that works on scalars, but you need to apply it to arrays. Writing loops everywhere clutters your code.

In [40]:
# Slow: Manual looping
def complex_transform(x):
    if x < 0:
        return np.sqrt(abs(x)) * -1
    else:
        return x ** 2

data = np.array([-4, 3, -1, 2])
results = np.array([complex_transform(x) for x in data])
results

array([-2.,  9., -1.,  4.])

Here’s the vectorized approach:

In [41]:
# Cleaner: np.vectorize()
def complex_transform(x):
    if x < 0:
        return np.sqrt(abs(x)) * -1
    else:
        return x ** 2

vec_transform = np.vectorize(complex_transform)
data = np.array([-4, 3, -1, 2])
results = vec_transform(data)
results

array([-2.,  9., -1.,  4.])

Here, `np.vectorize()` wraps your function so it can handle arrays. It automatically applies the function element-wise and handles the output array creation.  This doesn’t magically make your function faster. Under the hood, it’s still looping in Python. The advantage here is code clarity, not speed. For real performance gains, rewrite the function using NumPy operations directly:

In [42]:
# Actually fast
results = np.where(data < 0, -np.sqrt(np.abs(data)), data ** 2)
results

array([-2.,  9., -1.,  4.])

## <b><font color='darkblue'>6. np.einsum() for Complex Array Operations</font></b>
Matrix multiplications, transposes, traces, and tensor contractions pile up into unreadable chains of operations.

In [45]:
# Matrix multiplication the standard way
A = np.array([
    [1, 2],
    [3, 4],
])
B = np.array([
    [0],
    [1],
])
C = np.dot(A, B)
C

array([[2],
       [4]])

Here’s the vectorized approach:

In [46]:
C = np.einsum('ij,jk->ik', A, B)
C

array([[2],
       [4]])

In this example, `einsum()` uses [**Einstein summation notation**](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html). The string `'ij,jk->ik'` says: `“take indices i,j from the first array, j,k from the second, sum over shared index j, output has indices i,k.”`

## <b><font color='darkblue'>7. np.apply_along_axis() for Row/Column Operations</font></b>
When you need to apply a function to each row or column of a matrix, looping through slices works but feels clunky.

In [48]:
# Slow: Manual row iteration
data = np.array([
    [1, 2, 3],
    [4, 5, 6],
])

row_stats = []
for i in range(data.shape[0]):
    row = data[i]
    # Custom statistic not in NumPy
    stat = (np.max(row) - np.min(row)) / np.median(row)
    row_stats.append(stat)
    
row_stats = np.array(row_stats)
row_stats

array([1. , 0.4])

And here’s the vectorized approach:

In [49]:
def custom_stat(row):
    return (np.max(row) - np.min(row)) / np.median(row)
 
row_stats = np.apply_along_axis(custom_stat, axis=1, arr=data)
row_stats

array([1. , 0.4])

**In the above code snippet, `axis=1` means “apply the function to each row”** (<font color='brown'>axis 1 indexes columns, and applying along that axis processes row-wise slices</font>). The function receives 1D arrays and returns scalars or arrays, which get stacked into the result.

**Column-wise operations**: Use `axis=0` to apply functions down columns instead:

In [52]:
# Apply to each column
col_stats = np.apply_along_axis(custom_stat, axis=0, arr=data)
col_stats

array([1.2       , 0.85714286, 0.66666667])

## <b><font color='darkblue'>Wrapping Up</font></b>
<b><font size='3ptx'>Every technique in this article follows the same shift in thinking: describe what transformation you want applied to your data, not how to iterate through it.</font></b>

I suggest going through the examples in this article, adding timing to see how substantial the performance gains of using vectorized approaches are as compared to the alternative.

<b><font size='3ptx'>This isn’t just about speed. Vectorized code typically ends up shorter and more readable than its loop-based equivalent</font></b>. The loop version, on the other hand, requires readers to mentally execute the iteration to understand what’s happening. So yeah, happy coding!